In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
pre = pd.read_csv('pre_release_dataset_final_clean.csv')
post = pd.read_csv('post_release_dataset_clean_final.csv')
df = pd.merge(pre, post, on='game-name')

# Target engineering
df['log_steam_players'] = np.log1p(df['steam_current_players'])
df['log_youtube_views'] = np.log1p(df['youtube_total_views'])
df['success_score'] = 0.6 * df['log_steam_players'] + 0.4 * df['log_youtube_views']
df = df[df['success_score'] <= df['success_score'].quantile(0.99)]  # Remove top 1% outliers

# Aggressive grouping
def top_or_other(series, n=5):
    top = series.value_counts().index[:n]
    return series.apply(lambda x: x if x in top else 'Other')

df['steam_genres'] = df['steam_genres'].fillna('Unknown')
df['steam_developers'] = df['steam_developers'].fillna('Unknown')
df['steam_publishers'] = df['steam_publishers'].fillna('Unknown')

df['main_genre'] = top_or_other(df['steam_genres'], 5)
df['main_dev'] = top_or_other(df['steam_developers'], 5)
df['main_pub'] = top_or_other(df['steam_publishers'], 5)

# Count/interactions
df['n_genres'] = df['steam_genres'].apply(lambda x: len(str(x).split(';')))
df['dev_is_pub'] = (df['steam_developers'] == df['steam_publishers']).astype(int)

# Target encoding with KFold
def target_encode(train_X, train_y, test_X, col, n_splits=5):
    global_mean = train_y.mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_encoded = pd.Series(index=train_X.index, dtype=float)
    for train_idx, val_idx in kf.split(train_X):
        means = train_y.iloc[train_idx].groupby(train_X.iloc[train_idx][col]).mean()
        train_encoded.iloc[val_idx] = train_X.iloc[val_idx][col].map(means).fillna(global_mean)
    test_encoded = test_X[col].map(train_y.groupby(train_X[col]).mean()).fillna(global_mean)
    return train_encoded, test_encoded

# Prepare features/target
features = ['main_genre', 'main_dev', 'main_pub', 'n_genres', 'dev_is_pub']
X = df[features]
y = df['success_score']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Target encode categoricals
for col in ['main_genre', 'main_dev', 'main_pub']:
    tr_enc, te_enc = target_encode(X_train, y_train, X_test, col)
    X_train[f'{col}_te'] = tr_enc
    X_test[f'{col}_te'] = te_enc

# Drop original categoricals, keep encoded
X_train_final = X_train.drop(['main_genre', 'main_dev', 'main_pub'], axis=1)
X_test_final = X_test.drop(['main_genre', 'main_dev', 'main_pub'], axis=1)

# Try Ridge regression (linear)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_final, y_train)
ridge_pred = ridge.predict(X_test_final)

# Try LightGBM
lgbm = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
lgbm.fit(X_train_final, y_train)
lgbm_pred = lgbm.predict(X_test_final)

# Evaluate
def print_metrics(name, y_true, y_pred):
    print(f"{name} MAE: {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"{name} R²: {r2_score(y_true, y_pred):.4f}")

print_metrics("Ridge", y_test, ridge_pred)
print_metrics("LightGBM", y_test, lgbm_pred)

# Plot
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.scatter(y_test, ridge_pred, alpha=0.6, label='Ridge')
plt.scatter(y_test, lgbm_pred, alpha=0.6, label='LightGBM')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.legend()

plt.subplot(1,2,2)
plt.hist(y_test - ridge_pred, bins=30, alpha=0.5, label='Ridge')
plt.hist(y_test - lgbm_pred, bins=30, alpha=0.5, label='LightGBM')
plt.title('Residuals')
plt.legend()
plt.tight_layout()
plt.show()

KeyError: "None of [Index([ 5.509597829075253,  3.554211309391089, 2.4141925730099025,\n       2.9744111265407396,  6.689909759206893, 2.5770160665872797,\n        5.971591841247167,  4.906143961154019,  6.364661292437633,\n       2.9932977664295404,\n       ...\n        5.558758053664073,  5.765403887556778,  2.564069952786467,\n        2.584587270541487,  5.451540321915637, 0.2772588722239781,\n       1.8909551274849363, 1.2178089750893693,    4.7741515144429,\n       2.8936708718999395],\n      dtype='float64', length=933)] are in the [columns]"